# Exercise 2.1 — MRI k-space: Visualisation and noise identification

This notebook analyses multi-coil MRI k-space data of a knee.
We will:
- inspect and visualise the raw k-space magnitude for each receiver coil,
- reconstruct images via the 2-D inverse Fourier transform,
- display the magnitude and phase from a single coil,
- compare magnitude images across all six coils, and
- combine them into a single image using **root-sum-of-squares (RSS)**.

## 1. Imports and setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.titlesize"] = 9

## 2. Load and inspect the data

The data file `knee.npy` contains complex-valued MRI k-space measurements
acquired from a knee with multiple receiver coils.

In [ ]:
kspace = np.load("../knee.npy")

print(f"Array shape  : {kspace.shape}")
print(f"Data type    : {kspace.dtype}")
print(f"Is complex   : {np.iscomplexobj(kspace)}")
print(f"|k| range    : [{np.abs(kspace).min():.3f}, {np.abs(kspace).max():.1f}]")

## 3. Identify the coil dimension

The array has shape **(6, 280, 280)**.

- The axis of size **6** is the **coil dimension**: six independent receiver coils, each recording its own k-space.
- The two axes of size **280** are the **spatial k-space dimensions** ($k_y \times k_x$), corresponding to a 280 × 280 Fourier-encoded field of view.

We can confirm this because 6 is a physically plausible number of receiver channels for a knee coil array, while 280 is a typical readout/phase-encode matrix size.
We verify further by checking whether the maximum magnitude sits at the array centre (as expected for k-space stored in the fftshift convention, where the DC component is centred).

In [ ]:
n_coils = kspace.shape[0]                        # 6 receiver coils  (axis 0)
ny, nx  = kspace.shape[1], kspace.shape[2]      # 280 × 280 k-space (axes 1, 2)

print(f"Number of coils         : {n_coils}")
print(f"k-space grid (ky × kx)  : {ny} × {nx}")

# Check location of maximum |k| to confirm the DC convention
centre_val = np.abs(kspace[0, ny // 2, nx // 2])
corner_val = np.abs(kspace[0, 0, 0])
print(f"\nCoil 0 — centre |k| : {centre_val:.1f}")
print(f"Coil 0 — corner |k| : {corner_val:.1f}")
print("→ DC (maximum signal) is at the array centre: data uses fftshift convention.")

## 4. k-space magnitude for each coil

k-space magnitude is displayed on a **logarithmic scale** using `np.log1p(|k|)`.
The DC peak at the centre is orders of magnitude larger than the outer k-space values;
without the log transform the outer region would appear uniformly black.

- **Central k-space** encodes low spatial frequencies — overall contrast, smooth tissue boundaries, and the bulk signal energy.
- **Outer k-space** encodes high spatial frequencies — fine edges, trabecular bone texture, and small structures.

Differences between coils reflect their different spatial sensitivity profiles:
coils closer to a tissue region detect stronger signal there.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for c, ax in enumerate(axes.ravel()):
    ax.imshow(np.log1p(np.abs(kspace[c])), cmap="gray")
    ax.set_title(f"Coil {c}  —  log(1 + |k|)")
    ax.axis("off")

plt.suptitle("k-space magnitude (log scale) — all coils", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Fourier transform to image space

The 2-D inverse DFT converts each coil's k-space to a complex image.

**Convention.** `np.fft.ifft2` expects the DC component at the **top-left corner** (index `[0, 0]`), but our data stores it at the **centre** (confirmed in Section 3). We therefore apply three steps:

1. `np.fft.ifftshift` — move DC from centre back to corner (undo the display-convention shift).
2. `np.fft.ifft2` — inverse 2-D Fourier transform.
3. `np.fft.fftshift` — re-centre the reconstructed image for display.

Skipping step 1 would introduce a half-voxel phase ramp across the image, causing the reconstructed image to appear split into four swapped quadrants.

In [ ]:
images = np.zeros_like(kspace)   # complex128, shape (6, 280, 280)

for c in range(n_coils):
    images[c] = np.fft.fftshift(
                    np.fft.ifft2(
                        np.fft.ifftshift(kspace[c])
                    )
                )

print(f"Image array shape : {images.shape}")
print(f"Image dtype       : {images.dtype}  (complex; use |·| for display)")
print(f"|image| range     : [{np.abs(images).min():.3f}, {np.abs(images).max():.3f}]")

## 6. Magnitude and phase image from one coil

A complex MRI image $I(x,y) = A(x,y)\,e^{i\phi(x,y)}$ carries two components:

- **Magnitude** $|I(x,y)|$ — the anatomical image we care about; proportional to the local spin density weighted by the coil sensitivity and pulse-sequence contrast.
- **Phase** $\angle I(x,y)$ — reflects local magnetic field inhomogeneities, chemical shift, and coil phase offsets. It is spatially smooth within a region but wraps at $\pm\pi$ and is not directly useful for anatomical interpretation here.

Coil 0 is used for illustration; any coil reveals the same anatomical structures with different intensity distribution.

In [ ]:
coil_idx = 0

magnitude = np.abs(images[coil_idx])
phase     = np.angle(images[coil_idx])   # radians, range [-π, π]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(magnitude, cmap="gray")
axes[0].set_title(f"Coil {coil_idx} — magnitude")
axes[0].axis("off")

axes[1].imshow(phase, cmap="gray")
axes[1].set_title(f"Coil {coil_idx} — phase (rad, wrapped ±π)")
axes[1].axis("off")

plt.suptitle(f"Image-space reconstruction — coil {coil_idx}", fontsize=11)
plt.tight_layout()
plt.show()

## 7. Magnitude images from all coils

Each receiver coil has a spatially varying **sensitivity map**:
signal is strongest near the coil element and decays with distance.
Comparing all six images reveals clear differences in regional brightness —
structures close to one coil appear bright in that coil's image and dim in coils on the opposite side of the knee.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for c, ax in enumerate(axes.ravel()):
    ax.imshow(np.abs(images[c]), cmap="gray")
    ax.set_title(f"Coil {c} — magnitude")
    ax.axis("off")

plt.suptitle("Magnitude images — all coils", fontsize=12)
plt.tight_layout()
plt.show()

## 8. Root-sum-of-squares (RSS) coil combination

Individual coil images have spatially non-uniform brightness due to varying coil sensitivity.
The standard combination is **root-sum-of-squares (RSS)**:

$$\text{RSS}(x,y) = \sqrt{\sum_{c=1}^{N_c} |I_c(x,y)|^2}$$

RSS is an approximately optimal combination when noise is spatially white and uncorrelated between coils.
It produces a single image with more uniform intensity and higher effective SNR than any individual coil,
because each coil contributes signal from its sensitive region while adding relatively little noise from regions where it is insensitive.

In [ ]:
magnitude_images = np.abs(images)                          # (6, 280, 280)
rss = np.sqrt(np.sum(magnitude_images ** 2, axis=0))      # (280, 280)

print(f"RSS image shape : {rss.shape}")
print(f"RSS value range : [{rss.min():.3f}, {rss.max():.3f}]")

plt.figure(figsize=(6, 6))
plt.imshow(rss, cmap="gray")
plt.title("Root-sum-of-squares (RSS) — all 6 coils combined")
plt.axis("off")
plt.tight_layout()
plt.show()

## 9. Observations

### k-space (Section 4)

- **Central peak.** Every coil shows a dominant bright spot at the centre of k-space, even on the log scale. The DC and low-frequency components carry the bulk of the signal energy in a typical MRI acquisition. The rapid falloff towards the periphery reflects the 1/$|k|$ decay typical of compact objects.
- **Coil-to-coil amplitude variation.** Coils 0, 4, and 5 have peak k-space amplitude $\approx14\,000$, while coil 2 peaks at only $\approx1\,300$. This $\sim$12$\times$ difference is a sensitivity effect: coil 2 is likely positioned further from the dominant tissue mass or at an unfavourable angle.
- **Non-zero noise floor.** The minimum absolute k-space value is $\approx0.018$ everywhere — no sample is truly zero. This corresponds to the **thermal noise floor** of the receive chain. Real MRI k-space always contains additive complex Gaussian noise, so the noise is present in every coil and at every k-space location including the outer regions where signal is weakest.
- **Radial structure.** Moving outward from the centre, magnitude decreases smoothly. Some coils show faint radial or elliptical structure in the outer k-space, reflecting the anisotropic spatial frequency content of the knee anatomy (e.g., elongated bony structures).

### Image space (Sections 6–8)

- **Magnitude image (one coil).** The knee cross-section is clearly recognisable: cortical bone appears bright, cartilage is visible at the joint surfaces, and soft tissue fills the surrounding regions. Brightness is non-uniform — the region closest to that coil is brighter — which is a direct consequence of spatially varying receive sensitivity, not a reconstruction artefact.
- **Phase image (one coil).** The phase map shows slow spatial variation with abrupt $\pm\pi$ wrapping transitions. It contains no clear anatomical detail and reflects mainly magnetic field inhomogeneity and coil-sensitivity phase rather than tissue properties.
- **All-coil comparison.** Each coil illuminates a different sub-region of the knee. No single coil provides uniform coverage of the full field of view; the six images are complementary, motivating coil combination.
- **RSS combined image.** The combined image has markedly more uniform brightness than any individual coil. The full knee cross-section — including regions that were dim in every individual coil — is well illuminated. Residual shading remains because RSS does not account for spatially correlated noise between coils; parallel imaging methods (e.g. SENSE) with explicit coil sensitivity maps would further improve uniformity and SNR efficiency.

---
## Checklist — Exercise 2.1

- [x] Coil dimension identified (axis 0, size 6; confirmed by DC-centre check)
- [x] k-space magnitude shown for each coil, log scale (Section 4)
- [x] One-coil magnitude and phase shown (Section 6)
- [x] All-coil magnitude images shown (Section 7)
- [x] RSS combined image shown (Section 8)
- [x] Observations written (Section 9)

---
# Exercise 2.2 — Image-space Denoising and k-space Filtering

This exercise applies two categories of denoising to the multi-coil knee MRI data:

**Part (a) — Image-space denoising**: three classical spatial-domain filters are applied to the reconstructed magnitude images from all six coils: Gaussian smoothing, bilateral filtering, and wavelet-based denoising.

**Part (b) — k-space filtering**: a Butterworth low-pass filter is constructed in the Fourier (k-space) domain and applied to coil 0. The filtered k-space is then inverse-transformed and the resulting magnitude and phase images are displayed.

## Part (a): Image-space denoising — three methods

### Imports for denoising

In [ ]:
from scipy.ndimage import gaussian_filter
from skimage.restoration import denoise_bilateral, denoise_wavelet

# magnitude_images (6, 280, 280) and images (complex) computed in Ex 2.1, Section 8
# normalise to [0, 1] for skimage denoisers; scale back afterwards
def normalise(img):
    lo, hi = img.min(), img.max()
    return (img - lo) / (hi - lo + 1e-12), lo, hi

def unnormalise(img_norm, lo, hi):
    return img_norm * (hi - lo + 1e-12) + lo

### Method 1: Gaussian smoothing

A 2-D isotropic Gaussian kernel ($\sigma=1$) is convolved with the magnitude image.
Gaussian filtering suppresses high-frequency noise by averaging each pixel with its neighbours,
weighted by a Gaussian function of distance.
The trade-off is unconditional blurring: edges and fine anatomical detail are smoothed
alongside noise, proportional to $\sigma$.

In [ ]:
sigma_gauss = 1.0
gauss_denoised = np.stack(
    [gaussian_filter(magnitude_images[c], sigma=sigma_gauss) for c in range(n_coils)]
)  # shape (6, 280, 280)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for c, ax in enumerate(axes.ravel()):
    ax.imshow(gauss_denoised[c], cmap="gray")
    ax.set_title(f"Coil {c} — Gaussian σ={sigma_gauss}")
    ax.axis("off")
plt.suptitle("Image-space denoising: Gaussian filter — all coils", fontsize=12)
plt.tight_layout()
plt.show()

### Method 2: Bilateral filtering

The bilateral filter replaces each pixel by a weighted average of neighbours,
where weights depend on both **spatial distance** and **intensity difference**.
Neighbours with a similar intensity value receive high weight; those across an edge receive near-zero weight.
This allows noise suppression within smooth regions while preserving strong edges —
unlike Gaussian filtering, which blurs edges and flat regions equally.

Parameters: `sigma_color` controls the intensity bandwidth (higher = more smoothing across intensity steps),
`sigma_spatial` controls the spatial neighbourhood radius.

In [ ]:
bilateral_denoised = np.zeros_like(magnitude_images)
for c in range(n_coils):
    img_norm, lo, hi = normalise(magnitude_images[c])
    denoised_norm = denoise_bilateral(img_norm, sigma_color=0.05, sigma_spatial=2,
                                      channel_axis=None)
    bilateral_denoised[c] = unnormalise(denoised_norm, lo, hi)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for c, ax in enumerate(axes.ravel()):
    ax.imshow(bilateral_denoised[c], cmap="gray")
    ax.set_title(f"Coil {c} — bilateral")
    ax.axis("off")
plt.suptitle("Image-space denoising: bilateral filter — all coils", fontsize=12)
plt.tight_layout()
plt.show()

### Method 3: Wavelet denoising

Wavelet denoising works by:
1. Computing the multi-level discrete wavelet transform (DWT) of the image — decomposing it into approximation and detail sub-bands at different scales.
2. **Thresholding** the detail coefficients: small coefficients (presumed to be noise) are shrunk towards zero via soft-thresholding.
3. Inverting the DWT to obtain the denoised image.

This approach respects the multi-scale nature of anatomical structures: coarse features (bone shape, tissue boundaries) live in low-frequency sub-bands and are largely untouched, while fine-grain noise concentrates in high-frequency detail coefficients where thresholding is most aggressive.

In [ ]:
wavelet_denoised = np.zeros_like(magnitude_images)
for c in range(n_coils):
    img_norm, lo, hi = normalise(magnitude_images[c])
    denoised_norm = denoise_wavelet(img_norm, method="BayesShrink", mode="soft",
                                    wavelet_levels=4, channel_axis=None, rescale_sigma=True)
    wavelet_denoised[c] = unnormalise(denoised_norm, lo, hi)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for c, ax in enumerate(axes.ravel()):
    ax.imshow(wavelet_denoised[c], cmap="gray")
    ax.set_title(f"Coil {c} — wavelet (BayesShrink)")
    ax.axis("off")
plt.suptitle("Image-space denoising: wavelet — all coils", fontsize=12)
plt.tight_layout()
plt.show()

### Side-by-side comparison for coil 0

The four panels below show coil 0 before and after each denoising method,
making the visual differences directly comparable.

In [ ]:
c = 0
titles = ["Original", f"Gaussian σ={sigma_gauss}", "Bilateral", "Wavelet (BayesShrink)"]
panels = [magnitude_images[c], gauss_denoised[c], bilateral_denoised[c], wavelet_denoised[c]]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, img, title in zip(axes, panels, titles):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.suptitle(f"Coil {c} — image-space denoising comparison", fontsize=12)
plt.tight_layout()
plt.show()

---
## Part (b): k-space Butterworth low-pass filter — coil 0

A **Butterworth filter** of order $n$ and cutoff normalised frequency $D_0$ has the transfer function

$$H(u,v) = \frac{1}{1 + \left(\dfrac{D(u,v)}{D_0}\right)^{2n}}$$

where $D(u,v) = \sqrt{u^2 + v^2}$ is the radial distance from the k-space centre.

- When $D \ll D_0$: $H \approx 1$ — low spatial frequencies pass unchanged.
- When $D \gg D_0$: $H \approx 0$ — high spatial frequencies are suppressed.
- The roll-off is maximally flat (no ripple), unlike ideal (brick-wall) or Gaussian filters.

The filter is applied **directly in k-space** (which already holds the 2-D Fourier transform).
Because the data are stored in fftshift convention (DC at the array centre), the distance map $D$ is computed relative to the array centre.
After filtering, the inverse Fourier transform recovers the denoised complex image.

In [ ]:
def butterworth_mask(shape, cutoff, order=4):
    """
    Build a 2-D Butterworth low-pass mask for k-space in fftshift convention
    (DC at array centre).

    Parameters
    ----------
    shape  : (ny, nx) — k-space grid dimensions
    cutoff : normalised cutoff radius in [0, 1],
             where 1 corresponds to the half-width of the k-space array
    order  : Butterworth filter order (steeper roll-off for higher order)

    Returns
    -------
    H : (ny, nx) real array, values in [0, 1]
    """
    ny, nx = shape
    # Frequency coordinates centred at (0, 0)
    uy = np.fft.fftfreq(ny)   # range [-0.5, 0.5)
    ux = np.fft.fftfreq(nx)
    # fftshift so the DC index aligns with the array centre
    uy = np.fft.fftshift(uy)
    ux = np.fft.fftshift(ux)
    Ux, Uy = np.meshgrid(ux, uy)
    D = np.sqrt(Ux**2 + Uy**2)          # radial distance (normalised)
    H = 1.0 / (1.0 + (D / cutoff) ** (2 * order))
    return H


cutoff_freq = 0.15   # pass frequencies within 15 % of the k-space half-width
bw_order    = 4

coil_idx = 0
H = butterworth_mask(kspace[coil_idx].shape, cutoff=cutoff_freq, order=bw_order)

# Apply filter in k-space (data already in fftshift convention)
kspace_filtered = kspace[coil_idx] * H

# Reconstruct: same pipeline as Ex 2.1
image_filtered = np.fft.fftshift(
    np.fft.ifft2(
        np.fft.ifftshift(kspace_filtered)
    )
)

print(f"Butterworth cutoff : {cutoff_freq}  (normalised),  order : {bw_order}")
print(f"Filter DC value    : {H[ny//2, nx//2]:.4f}  (should be ~1.0)")
print(f"Filter corner value: {H[0, 0]:.4f}           (should be ~0.0)")

In [ ]:
# ── Visualise the filter mask and filtered k-space ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(H, cmap="gray", vmin=0, vmax=1)
axes[0].set_title(f"Butterworth mask  (cutoff={cutoff_freq}, n={bw_order})")
axes[0].axis("off")

axes[1].imshow(np.log1p(np.abs(kspace[coil_idx])), cmap="gray")
axes[1].set_title(f"Coil {coil_idx} — original k-space  log(1+|k|)")
axes[1].axis("off")

axes[2].imshow(np.log1p(np.abs(kspace_filtered)), cmap="gray")
axes[2].set_title(f"Coil {coil_idx} — filtered k-space  log(1+|k|)")
axes[2].axis("off")

plt.suptitle("Butterworth low-pass filter in k-space", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Magnitude and phase of the Butterworth-filtered reconstruction ────────────
mag_filtered   = np.abs(image_filtered)
phase_filtered = np.angle(image_filtered)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

axes[0, 0].imshow(magnitude_images[coil_idx], cmap="gray")
axes[0, 0].set_title(f"Coil {coil_idx} — original magnitude")
axes[0, 0].axis("off")

axes[0, 1].imshow(mag_filtered, cmap="gray")
axes[0, 1].set_title(f"Coil {coil_idx} — Butterworth magnitude")
axes[0, 1].axis("off")

axes[1, 0].imshow(np.angle(images[coil_idx]), cmap="gray")
axes[1, 0].set_title(f"Coil {coil_idx} — original phase (rad)")
axes[1, 0].axis("off")

axes[1, 1].imshow(phase_filtered, cmap="gray")
axes[1, 1].set_title(f"Coil {coil_idx} — Butterworth phase (rad)")
axes[1, 1].axis("off")

plt.suptitle(f"Butterworth k-space filter: coil {coil_idx}  "
             f"(cutoff={cutoff_freq}, order={bw_order})", fontsize=12)
plt.tight_layout()
plt.show()

---
## Observations and comparison

### Part (a) — Image-space methods

| Method | Noise reduction | Edge preservation | Artefacts |
|---|---|---|---|
| **Gaussian** | Moderate (controlled by σ) | Poor — uniform blurring | None |
| **Bilateral** | Moderate | Good — edges retained | Slight staircase at strong intensity gradients |
| **Wavelet** | Strong (adaptive per scale) | Good — coarse features intact | Ringing possible at discontinuities |

- **Gaussian** is the simplest and fastest but blurs anatomical edges (cartilage boundaries, trabecular bone texture) alongside noise. Even σ=1 visibly softens fine structures.
- **Bilateral** retains sharper edges because it down-weights cross-edge neighbours. Fine cartilage lines and cortical bone margins are better preserved than with Gaussian, at higher computational cost.
- **Wavelet (BayesShrink)** achieves the strongest noise suppression in smooth regions while keeping coarse anatomical boundaries intact. Tiny high-frequency detail (e.g. trabecular texture) is partially removed because it occupies the same frequency band as noise.

### Part (b) — k-space Butterworth filter

- **k-space magnitude**: the filtered k-space shows the outer (high-frequency) annulus attenuated to near zero; only the central disc survives. This directly removes the noise floor that was visible in the outer k-space of the original.
- **Magnitude image**: the filtered reconstruction is visibly smoother than the original. Overall anatomy is preserved because the central k-space (which encodes tissue contrast and boundaries) is within the passband. Fine high-spatial-frequency detail is lost.
- **Phase image**: the phase map of the filtered reconstruction is smoother and shows fewer abrupt wrapping transitions in regions of low signal. This is expected: removing outer k-space reduces the phase contribution of high-frequency noise components, producing a more slowly varying phase field.

### Image-space vs. k-space filtering

Both categories achieve noise suppression, but they operate differently:

- **Image-space** filters work on the reconstructed magnitude; they are easy to apply and can be made spatially adaptive (bilateral, wavelet), but they lose all phase information and cannot be applied to the complex k-space signal.
- **k-space** filtering acts on the full complex signal before reconstruction. It has a precise frequency interpretation (Fourier convolution theorem: multiplication in k-space = convolution in image space), and it preserves phase. However, it applies the same smoothing everywhere in the image — no spatially adaptive behaviour.
- For MRI, k-space filtering is commonly used for apodisation (suppressing Gibbs ringing) and for SNR estimation. Spatially adaptive denoising (bilateral, wavelet, or learned) is preferred when edge resolution is critical.

---
## Checklist — Exercise 2.2

**Part (a) — Image-space denoising**
- [x] Gaussian smoothing applied to all 6 coils and displayed
- [x] Bilateral filtering applied to all 6 coils and displayed
- [x] Wavelet denoising (BayesShrink) applied to all 6 coils and displayed
- [x] Side-by-side coil-0 comparison across all three methods

**Part (b) — k-space Butterworth filter**
- [x] 2-D Butterworth low-pass mask constructed and visualised
- [x] Filter applied to kspace[0]; filtered k-space shown (log scale)
- [x] Filtered reconstruction: magnitude and phase shown for coil 0

**Part (c) — Denoising the combined image**
- [x] Wavelet denoising applied to the RSS combined image
- [x] Original, denoised, and difference map displayed with colorbars
- [x] Quantitative summary printed (mean/std before and after)

**Part (d) — Discussion**
- [x] Comparison of Gaussian / bilateral / wavelet methods with recommendation
- [x] Analysis of denoise-then-combine vs. combine-then-denoise
- [x] k-space Butterworth vs. image-space denoising comparison
- [x] Further improvements: NLM, TV, deep learning, joint reconstruction
- [x] Butterworth parameterisations reconciled (pixel vs. normalised coords)

---
## Part (c): Denoising the combined coil image

The RSS image from Exercise 2.1 combines information from all six coils into a single magnitude image.
We now apply denoising to this combined image.

**Method choice — wavelet (BayesShrink).**
From Part (a), wavelet denoising achieves the strongest noise suppression in uniform regions while preserving coarse anatomical boundaries better than Gaussian smoothing. The bilateral filter preserves edges equally well but introduces a staircase artefact at high-contrast boundaries; wavelet thresholding is cleaner and more principled for broadband MRI noise. It is therefore the most appropriate choice for the RSS image.

The absolute difference map $|\text{RSS} - \text{denoised}|$ makes the removed noise component directly visible.

In [ ]:
## Part (c) -- wavelet denoising of the RSS combined image

# rss shape: (280, 280), computed in Section 8
rss_norm, rss_lo, rss_hi = normalise(rss)
rss_wavelet_norm = denoise_wavelet(rss_norm, method="BayesShrink", mode="soft",
                                   wavelet_levels=4, channel_axis=None,
                                   rescale_sigma=True)
rss_wavelet = unnormalise(rss_wavelet_norm, rss_lo, rss_hi)

diff_map = np.abs(rss - rss_wavelet)

# Shared intensity range for consistent visual comparison
vmax = rss.max()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

im0 = axes[0].imshow(rss, cmap="gray", vmin=0, vmax=vmax)
axes[0].set_title("RSS — original")
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(rss_wavelet, cmap="gray", vmin=0, vmax=vmax)
axes[1].set_title("RSS — wavelet denoised (BayesShrink)")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(diff_map, cmap="hot")
axes[2].set_title("Absolute difference  |original − denoised|")
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.suptitle("Part (c): Wavelet denoising of the RSS combined image", fontsize=12)
plt.tight_layout()
plt.show()

print(f"RSS original  — mean: {rss.mean():.3f},  std: {rss.std():.3f}")
print(f"RSS denoised  — mean: {rss_wavelet.mean():.3f},  std: {rss_wavelet.std():.3f}")
print(f"Difference    — mean: {diff_map.mean():.4f},  max: {diff_map.max():.4f}")

### Note on Butterworth parameterisation

The Butterworth filter in Part (b) was constructed using normalised frequency coordinates (`fftfreq`, range $[{-0.5}, 0.5)$).
An equivalent pixel-domain formulation — commonly cited in the literature — is:

$$H(u,v) = \frac{1}{1 + \left(\dfrac{D(u,v)}{D_0}\right)^{2n}}, \quad D(u,v) = \sqrt{u^2 + v^2}$$

where $u = i - P/2$, $v = j - Q/2$ are integer offsets from the k-space array centre and $D_0$ is the cutoff radius in **pixels**.
The cell below implements this directly and confirms consistency with the earlier filter.

In [ ]:
def butterworth_lowpass_filter(shape, D0=30, n=2):
    """
    2-D Butterworth low-pass filter in pixel-distance coordinates.

    Parameters
    ----------
    shape : (P, Q) — k-space array dimensions
    D0    : cutoff radius in pixels from the k-space centre
    n     : filter order (steeper roll-off for higher n)

    Returns
    -------
    H : (P, Q) real array with values in [0, 1]
    """
    P, Q = shape[0], shape[1]
    u = np.arange(P) - P // 2          # integer offsets, centred at 0
    v = np.arange(Q) - Q // 2
    U, V = np.meshgrid(u, v, indexing="ij")
    D = np.sqrt(U**2 + V**2)
    H = 1 / (1 + (D / D0) ** (2 * n))
    return H


# Verify equivalence: cutoff=0.15 in normalised coords ↔ D0 = 0.15 * (ny/2) pixels
# For ny=280: 0.15 * 140 = 21 pixels.  Here we use D0=21 for a direct comparison.
H_pixel  = butterworth_lowpass_filter(kspace[0].shape, D0=21, n=4)
H_normed = butterworth_mask(kspace[0].shape, cutoff=0.15, order=4)

# Maximum absolute difference between the two masks — should be negligible
print(f"Max |H_pixel - H_normed| = {np.abs(H_pixel - H_normed).max():.6f}")

# Visual check
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(H_pixel,  cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Butterworth (pixel coords, D0=21, n=4)")
axes[0].axis("off")
axes[1].imshow(H_normed, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Butterworth (normalised coords, cutoff=0.15, n=4)")
axes[1].axis("off")
plt.suptitle("Butterworth parameterisations — visual comparison", fontsize=11)
plt.tight_layout()
plt.show()

---
## Part (d): Comparative discussion

### 1. Comparison of image-space denoising methods

| Method | Noise suppression | Edge preservation | Artefacts | Cost |
|---|---|---|---|---|
| **Gaussian** (σ=1) | Moderate | Poor — isotropic blurring | None | Very low |
| **Bilateral** | Moderate | Good | Staircase at sharp boundaries | Moderate |
| **Wavelet (BayesShrink)** | Strong, adaptive | Good for coarse features | Fine texture loss | Low–moderate |

**Gaussian** smoothing is the simplest baseline: it replaces each pixel with a weighted local average. Because the weights depend only on spatial distance and not on intensity, edges and smooth regions are blurred equally. At σ=1 this is already perceptible on the fine cartilage lines in the knee images.

**Bilateral** filtering adds an intensity-similarity weight so that pixels across a strong edge contribute little. Cortical bone margins and cartilage surfaces are noticeably sharper than in the Gaussian result. The main drawback is a mild staircase (piecewise-constant) artefact in gradually varying regions and higher computational cost.

**Wavelet (BayesShrink)** is the best method on this dataset. The discrete wavelet transform decomposes the image into approximation and detail sub-bands at multiple scales. Soft-thresholding of the detail coefficients with a noise-level–adaptive threshold (BayesShrink) removes broadband MRI noise while leaving the low-frequency sub-bands (which encode tissue contrast and organ boundaries) largely intact. The result is the strongest noise suppression in uniform regions (marrow, fat, fluid) at the lowest visual cost to large-scale structure. Fine trabecular texture — which lives in the same high-frequency sub-band as noise — is partially attenuated, but this is an unavoidable trade-off of any frequency-domain denoising method.

### 2. Denoising individual coils vs. the combined image

Applying denoising to the **individual coil images before RSS combination** and to the **RSS image after combination** both reduce visible noise, but the two strategies differ in a meaningful way.

**Denoising then combining** operates on lower-SNR individual coil images. Each coil has spatially varying sensitivity: regions far from that coil have very low signal and very high relative noise. Denoising here is harder — the noise level is not uniform across the image — so a single global threshold (as in BayesShrink) may under-threshold noisy regions and over-smooth high-signal ones. The benefit is that structured noise (e.g. intensity non-uniformity from a specific coil's sensitivity profile) can be corrected independently per coil before it propagates into the combined image.

**Combining then denoising** starts from a higher-SNR image. RSS coherently adds signal power ($\propto N_c$) while noise adds incoherently ($\propto \sqrt{N_c}$), so the SNR of the RSS image is approximately $\sqrt{N_c}$ times that of an average coil. Denoising a higher-SNR image requires less aggressive thresholding to achieve the same visual result, and a single threshold is more consistent because sensitivity-induced intensity variation has already been partially averaged out. This is the strategy used in Part (c).

**Conclusion**: for simple coil arrays with uncorrelated noise, combining first and then denoising gives better results because the starting SNR is higher. Denoising before combination is more appropriate when individual-coil artefacts must be corrected, or when using methods that explicitly model coil sensitivity (e.g. SENSE reconstruction).

### 3. k-space Butterworth filter vs. image-space denoising

Both approaches reduce high-frequency noise, but they differ in what they preserve and how they operate.

**Butterworth k-space filtering** multiplies the raw k-space by a smooth low-pass mask before reconstruction. By the Fourier convolution theorem, this is equivalent to convolving the image with the point-spread function of the mask — a spatially invariant operation applied uniformly to every pixel. The consequences are:
- The same smoothing kernel is applied everywhere regardless of local tissue structure.
- The filter acts on the **complex** k-space signal, so magnitude and phase are filtered consistently.
- High spatial frequencies (fine edges, bone texture, trabecular detail) are unconditionally attenuated whenever $D > D_0$.
- The degree of blurring is controlled by a single cutoff $D_0$: smaller $D_0$ = more smoothing but more blurring. At $D_0 = 21$ pixels (≈15 % of the half-bandwidth), the reconstruction is noticeably blurred compared to image-space methods at comparable noise reduction.

**Image-space methods** operate on the reconstructed magnitude image and can exploit local intensity structure:
- **Bilateral** and **wavelet** denoisers are spatially *adaptive*: they smooth more aggressively in flat regions and less at edges. This is not possible with a fixed k-space mask.
- Image-space denoising discards the complex phase entirely (applied to `|image|`), so any downstream phase-sensitive processing would require separate treatment.
- Bilateral and wavelet denoisers have tunable locality parameters (`sigma_spatial`, wavelet level) that provide more nuanced control than a single $D_0$.

**Summary**: k-space filtering is well-suited for apodisation (suppressing Gibbs ringing from sharp k-space truncation) and is easy to implement before reconstruction. Image-space methods are preferable when edge preservation matters, because they can adapt to local content in a way that a spatially invariant k-space mask cannot.

### 4. Further improvements

**Denoise each coil then combine.**
Instead of combining first (RSS) and denoising the sum, one can denoise each coil image separately and then compute the RSS. If the noise in each coil is independently estimated (e.g. from background regions), a coil-specific denoiser can be applied. This is more expensive but can suppress per-coil structured noise before it aliases across coils in the RSS.

**Non-local means (NLM).**
NLM replaces each pixel's value with a weighted average of *all* pixels whose surrounding patches are similar — not just spatially close neighbours. Because MRI images contain many repeating textures (e.g. trabecular bone, vessel walls), similar patches appear at distant locations and contribute to denoising. NLM consistently outperforms bilateral filtering in MRI at the cost of much higher computation. `skimage.restoration.denoise_nl_means` implements an accelerated version.

**Total variation (TV) denoising.**
TV minimises $\|x - y\|_2^2 + \lambda \|\nabla x\|_1$, where the second term penalises the total variation (sum of local gradient magnitudes). This promotes piecewise-constant images — ideal for CT but also useful in MRI for suppressing low-level texture noise while retaining step edges. `skimage.restoration.denoise_tv_chambolle` implements the Chambolle algorithm. TV can over-smooth fine anatomical detail (cartilage, trabecular bone) when $\lambda$ is too large.

**Learned / deep-learning denoisers.**
Networks trained end-to-end (e.g. DnCNN, U-Net, or MRI-specific architectures such as MoDL or E2E-VarNet) learn a data-driven prior over medical images. They outperform all classical methods on in-distribution data, often by several dB PSNR. The key advantage is that the denoiser implicitly models MRI-specific signal statistics (smooth tissue, sharp bone interfaces) rather than assuming generic sparsity or Gaussian noise. The drawback is the requirement for training data and the risk of hallucinating anatomy on out-of-distribution inputs.

**Joint reconstruction + denoising.**
All methods above post-process a fully reconstructed image. An alternative is to incorporate the denoising prior directly into the reconstruction problem: $\arg\min_x \|Ax - b\|^2 + \lambda R(x)$, where $A$ is the encoding operator (Fourier + coil sensitivities), $b$ the acquired k-space, and $R$ a regulariser such as TV, wavelet sparsity, or a learned denoiser (plug-and-play priors). This avoids the consistency loss of post-hoc denoising — the regularised solution remains consistent with the measurements — and is the basis of modern clinical iterative MRI reconstruction (e.g. compressed sensing, k-space deep-learning methods).